[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/saghiles/semantic-id-tutorial/blob/main/session2_cross_domain_transfer.ipynb)

# Hands-on 2: Cross-Domain Transfer with Semantic IDs

In Session 1 we trained a generative recommender (**TIGER**) on Amazon *Beauty*. Every item was represented by a **Semantic ID**: a short tuple of discrete codes obtained by quantizing the item's *content embedding* (residual k-means), plus a final *dedup* token that separates items sharing the same codes.

Because Semantic IDs are derived from content rather than learned per item, a trained model can in principle be **reused on a new catalog by re-tokenizing it**, with no gradient updates. This notebook tests that idea.

### What we will do
1. Load the Session 1 TIGER model (the **source** model) and evaluate it in-domain.
2. Carve out a **target domain** from Beauty: the *Skin Care* sub-category.
3. Embed the target items with the *same* text encoder, and quantize them with the source model's **frozen** residual k-means centroids.
4. Plug the new Semantic IDs into the **frozen** source transformer and evaluate next-item prediction (`TIGERTransfer`).
5. Take-home exercises: handling higher collision rates, and a true cold-start experiment.

## 1. Setup

In [ ]:
import os, sys

# --- Colab setup (skipped automatically when running locally) ---

if "google.colab" in sys.modules:
    REPO_URL = "https://github.com/saghiles/semantic-id-tutorial.git"
    REPO_DIR = "/content/semantic-id-tutorial"
    # the item features / saved models are stored with Git LFS
    !command -v git-lfs >/dev/null || apt-get install -y -qq git-lfs
    !git lfs install
    if not os.path.exists(REPO_DIR):     # re-running this cell is safe
        # clone the pointer files first, then fetch the real LFS files so any
        # LFS download error is shown instead of silently aborting the checkout
        !GIT_LFS_SKIP_SMUDGE=1 git clone {REPO_URL} {REPO_DIR}
    %cd {REPO_DIR}
    !git lfs pull
    !pip install -q cornac==3.0.0
    !ls -lh item_features

import re
from collections import Counter, defaultdict

import numpy as np
import torch
from sentence_transformers import SentenceTransformer

from cornac.data import FeatureModality
from cornac.datasets import amazon_review
from cornac.eval_methods import NextItemEvaluation
from cornac.experiment import Experiment
from cornac.metrics import MRR, NDCG, Recall
from cornac.models import TIGER
from cornac.models.recommender import NextItemRecommender

In [ ]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

SAVE_DIR = "saved_models"                                   # where Session 1 saved its model
SOURCE_MODEL_PATH = os.path.join(SAVE_DIR, "TIGER")

SPLIT_SEED = 123                                            # same seed as at training time
ENCODER_NAME = "sentence-t5-base"                           # MUST match the encoder used for the source model
TARGET_CATEGORY = "Skin Care"                               # 2nd-level Amazon category used as target domain
TARGET_FEATURE_CACHE = "item_features/amazon_beauty_skincare"       # avoids re-encoding on re-runs

metrics = [Recall(k=10), NDCG(k=10), MRR()]
print("Device:", DEVICE)

## 2. Helper functions

### 2.1 Quantizing with *frozen* residual k-means
At training time, residual k-means learned one codebook per level. To tokenize **new** items we do not re-fit anything: at each level we assign the nearest centroid, subtract it, and pass the residual to the next level. This is exactly the encoding logic of the source model, applied to new embeddings.

In [ ]:
@torch.no_grad()
def rkmeans_encode(feats, centroids, device="cpu", chunk=65536):
    """Quantize (N, D) feats with FROZEN residual-kmeans centroids -> (N, L) int64."""
    cents = [torch.as_tensor(c, device=device) for c in centroids]
    out = []
    for s in range(0, len(feats), chunk):
        r = torch.as_tensor(feats[s:s + chunk], device=device)
        codes = []
        for C in cents:                      # C: (K, D)
            ids = torch.cdist(r, C).argmin(dim=1)
            r = r - C[ids]                   # same residual logic as _fit_rkmeans
            codes.append(ids)
        out.append(torch.stack(codes, 1).cpu())
    return torch.cat(out).numpy().astype("int64")

### 2.2 Collision diagnostics
Two items *collide* when they receive the same semantic codes (all levels except the last one). The last level (the **dedup token**) is what makes the full ID unique, and its vocabulary size is fixed in the source model. If the target domain produces larger collision groups than the source can accommodate, transfer breaks, which is the subject of Exercise 1.

In [ ]:
def collision_report(name, sid_table):
    """sid_table: (N, L+1) array, where the last column is the dedup index."""
    prefix = np.asarray(sid_table)[:, :-1]
    _, inverse, counts = np.unique(prefix, axis=0, return_inverse=True, return_counts=True)
    group_size = counts[inverse.ravel()]
    n = len(prefix)
    print(f"[{name}] items: {n} | distinct code prefixes: {len(counts)} | "
          f"items sharing a prefix: {(group_size > 1).sum()} ({100 * (group_size > 1).mean():.1f}%) | "
          f"largest collision group (= dedup slots needed): {counts.max()}")

## 3. The transfer model

`TIGERTransfer` subclasses `TIGER` but **trains nothing**. Its `fit()` only:

| Step | What happens |
|------|--------------|
| 1 | Copies the frozen transformer, device and structural hyper-parameters (`rqvae_num_levels`, `rqvae_codebook_size`, `max_len`) from the source. |
| 2 | Gets target-domain text embeddings and (optionally) standardizes them. |
| 3 | Quantizes them with the **source** residual k-means centroids. |
| 4 | Builds the target Semantic ID table (including dedup tokens), prefix tries for constrained beam search, and the SID → item map. |
| 5 | Checks that the target's dedup range fits in the source's dedup vocabulary, then builds the encoder token table using the source's token **offsets**. |

**Design note on feature standardization.** If the source model standardized its features, we must do the same for the target. We can use the *target* statistics (default, no extra information needed) or pass the *source* statistics via `feat_mean` / `feat_std` (keeps the embedding space identical to the one the centroids were fit in, but requires embedding the source catalog). The choice is worth experimenting with.

In [ ]:
class TIGERTransfer(TIGER):
    """Frozen source model; fit() only re-tokenizes the target catalog."""

    def __init__(self, source, feat_mean=None, feat_std=None, name="TIGER-transfer", **kw):
        super().__init__(name=name, trainable=False, **kw)
        self.src = source
        self.feat_mean, self.feat_std = feat_mean, feat_std   # optional SOURCE statistics

    def fit(self, train_set, val_set=None):
        if hasattr(self, "model") and self.model is not None:
            return self  # already fitted
        NextItemRecommender.fit(self, train_set, val_set)     # sets train_set, total_items
        src = self.src

        # 1) reuse the frozen source model and its structural hyper-parameters
        for a in ("rqvae_num_levels", "rqvae_codebook_size", "max_len"):
            setattr(self, a, getattr(src, a))
        self.model, self.device_ = src.model, src.device_

        # 2) target-domain embeddings (+ standardization, if the source used it)
        feats = self._get_item_features()
        if src.feature_standardize:
            mean = feats.mean(axis=0) if self.feat_mean is None else self.feat_mean
            std = feats.std(axis=0) if self.feat_std is None else self.feat_std
            std = np.where(std == 0, 1.0, std)
            feats = ((feats - mean) / std).astype("float32")

        # 3) quantize with FROZEN source centroids
        codes = rkmeans_encode(feats, src.rkmeans_centroids, device=self.device_)
        self.target_codes = codes                              # kept for diagnostics

        # 4) semantic ids, tries, sid_to_item (+ dedup level)
        self._build_semantic_ids(codes)

        # 5) the dedup vocabulary of the source is fixed: the target must fit in it
        print(f"Source model has {src.level_sizes[-1]} collision slots, "
              f"target needs {self.level_sizes[-1]}.")
        if self.level_sizes[-1] > src.level_sizes[-1]:
            raise ValueError(
                f"Target needs {self.level_sizes[-1]} collision slots, "
                f"source model has {src.level_sizes[-1]}. Increase reserve_dedup.")
        self.level_sizes = list(src.level_sizes)

        offsets = src.model.offsets.cpu().numpy()
        self.pad_idx = self.total_items
        self.enc_token_table = np.zeros((self.total_items + 1, len(self.level_sizes)), "int64")
        self.enc_token_table[: self.total_items] = self.sid_table + offsets
        return self

## 4. Source domain: Amazon Beauty

We load the feedback (`user, item, rating, timestamp`) and the item texts, then load the TIGER model trained in Session 1 (`trainable=False`: we will never update it).

In [ ]:
data = amazon_review.load_feedback(category="beauty")           # (user, item, rating, timestamp)
texts, item_ids = amazon_review.load_text(category="beauty")    # item text, aligned with item_ids
id2text = dict(zip(item_ids, texts))
print(f"{len(data)} interactions | {len(item_ids)} items with text")

In [ ]:
tiger = TIGER.load(SOURCE_MODEL_PATH, trainable=False)
print(f"Loaded TIGER ({tiger.num_items} items, {len(tiger.sid_table)} SIDs)")
collision_report("source / Beauty", tiger.sid_table)

## 5. Target domain: Skin Care

Each item text ends with a `Categories:` field holding the category path (e.g. `Beauty, Skin Care, Face, ...`). The **second level** gives us the sub-categories of Beauty, and we use one of them as the target domain.

In [ ]:
CATEGORY_RE = re.compile(r"Categories:\s*(.*)$", flags=re.DOTALL)

def second_level_category(text):
    """Return the 2nd level of the category path (e.g. 'Skin Care'), or None."""
    m = CATEGORY_RE.search(text)
    if not m:
        return None
    levels = [c.strip() for c in m.group(1).split(",")]
    return levels[1] if len(levels) >= 2 and levels[1] else None

item_category = {i: second_level_category(t) for i, t in zip(item_ids, texts)}
category_counts = Counter(c for c in item_category.values() if c)
print(dict(category_counts.most_common()))

In [ ]:
# Target catalog: texts and ids kept aligned
target_pairs = [(i, t) for i, t in zip(item_ids, texts) if item_category[i] == TARGET_CATEGORY]
target_ids, target_texts = (list(x) for x in zip(*target_pairs))
target_id_set = set(target_ids)

# Target interactions: keep only interactions with target items
target_data = [row for row in data if row[1] in target_id_set]
print(f"Target category: {TARGET_CATEGORY} | items: {len(target_ids)} | interactions: {len(target_data)}")

## 6. Embed the target items

**Important:** the encoder must be *the same model* that produced the source embeddings. The frozen centroids only make sense in that embedding space.

In [ ]:
if os.path.exists(f"{TARGET_FEATURE_CACHE}.npz"):
    cache = np.load(f"{TARGET_FEATURE_CACHE}.npz")
    target_features = cache['features']
    cached_target_ids = cache['item_ids']
    assert np.array_equal(cached_target_ids, np.array(target_ids)), "Cached target ids do not match current target ids."    
    assert len(target_features) == len(target_ids), "Stale cache: delete it and re-encode."
    print(f"Loaded cached target features -> {TARGET_FEATURE_CACHE}  {target_features.shape}")
else:
    encoder = SentenceTransformer(ENCODER_NAME, device=DEVICE)
    target_features = encoder.encode(target_texts, batch_size=128, show_progress_bar=True)
    np.savez(f"{TARGET_FEATURE_CACHE}.npz", features=target_features, item_ids=np.array(target_ids))
    del encoder
    if DEVICE == "cuda":
        torch.cuda.empty_cache()

print(target_features.shape)

## 7. Target data split
Same leave-last-out protocol and seed, but applied to the Skin Care interactions only. Users with fewer than 3 Skin Care interactions are dropped, and the item embeddings are attached as an item feature modality.

In [ ]:
next_item_eval_target = NextItemEvaluation.leave_last_out(
    data=target_data,
    exclude_unknowns=True,
    verbose=True,
    item_feature=FeatureModality(features=target_features, ids=target_ids),
    seed=SPLIT_SEED,
)

## 8. Transfer
We wrap the frozen source model in `TIGERTransfer` and evaluate. `n_beams` controls the beam width of the constrained generation.

In [ ]:
target_tiger = TIGERTransfer(tiger, n_beams=30, verbose=True)

Experiment(
    eval_method=next_item_eval_target,
    models=[target_tiger],
    metrics=metrics,
    show_validation=False,
).run()

## 9. Key takeaways

* Semantic IDs are **content-derived**, so a catalog from a new domain can be tokenized with *frozen* codebooks and fed to an already-trained generative recommender: no new item-embedding table, no retraining.
* The transferable pieces are the transformer weights and the token vocabulary; the part that must be re-computed is the *item → SID* mapping.
* The dedup (collision) level is the fragile part: its vocabulary size is fixed at training time and bounds how many near-identical items the target may contain.
* The source and target feature/content distributions must match 

---
# Take-home exercises

## Exercise 1: Handle a higher collision rate in the target domain
The current `TIGERTransfer.fit()` raises a `ValueError` as soon as the target needs more dedup slots than the source model reserved. Improve the implementation so that transfer still works when the target domain has a **higher collision rate** than the source.

## Exercise 2: Cold-start transfer (source model never sees Skin Care)
In this notebook the source model had already been trained on Skin Care items. Build a *real* cold-start scenario:

1. **Retrain** a source TIGER model on Amazon Beauty **without** the Skin Care category (remove both the items and their interactions).
2. **Evaluate** this model on the Skin Care target domain through `TIGERTransfer`, using the *same* target split (`next_item_eval_target`) as above.
3. **Compare** (same metrics, same split):
   * the transfer model from Session 1 (saw Skin Care during training),
   * the cold-start transfer model (never saw Skin Care)

Tips: reuse the *exact* hyper-parameters and encoder of Session 1 so that the comparison is fair..

In [ ]:
# Exercise 2, starter: source data WITHOUT the target category
non_target_items = {i for i in item_ids if item_category[i] != TARGET_CATEGORY}
data_no_target = [row for row in data if row[1] in non_target_items]
texts_no_target = [id2text[i] for i in item_ids if i in non_target_items]
print(f"Source data without {TARGET_CATEGORY}: {len(non_target_items)} items, {len(data_no_target)} interactions")

# TODO 1: embed texts_no_target with the same encoder and split data_no_target (same protocol/seed)
# TODO 2: train TIGER with the Session 1 hyper-parameters; save to "saved_models/TIGER_no_skincare"
# TODO 3: tiger_cold = TIGER.load(..., trainable=False); TIGERTransfer(tiger_cold, n_beams=30)
# TODO 4: run Experiment(eval_method=next_item_eval_target, models=[...], metrics=metrics)